# Per-Muscle and Overall Average Metrics — Augmented Dataset

For each algorithm that has a `gpu_lambda_column_compare_*_augmented.ipynb`,
this notebook reads the per-muscle-group result CSVs from
`{algo}/codes/results_augmented/`, computes the mean of every metric across
all 20 augmented volumes, appends an `Overall_Mean` row, and saves a summary
CSV. The final cell combines all `Overall_Mean` rows into one comparison
table with colour-coded Dice/Hausdorff columns and the best value per metric
bolded.

Unlike the Sheffield evaluation (37 fine-grained muscle labels), the
augmented dataset's ground truth only distinguishes 4 bilateral muscle
groups (myosegmenTUM `combined_gt` scheme): **Gracilis, Hamstrings,
Quadriceps, Sartorius**. Some algorithms (MedCLIP-SAMv2, MedCLIP-SAMv2
Text+Boxes, MedSegDiff) only predict Gracilis/Sartorius on this dataset, so
their rows for Hamstrings/Quadriceps are naturally absent.

In [28]:
import pathlib
import re
import warnings
import numpy as np
import pandas as pd
from IPython.display import display

In [29]:
# ── Paths ──────────────────────────────────────────────────────────────────────
EVAL_DIR    = pathlib.Path(r'C:\Projects\dissector\eval_notebooks')
SUMMARY_DIR = EVAL_DIR / 'summary_results_augmented'
SUMMARY_DIR.mkdir(exist_ok=True)

# ── Augmented-dataset muscle groups (myosegmenTUM combined_gt scheme) ───────────
AUGMENTED_MUSCLES = ['gracilis', 'hamstrings', 'quadriceps', 'sartorius']

# Regex to strip '{muscle_name}_' prefix from column names
_AUG_PREFIX_RE = re.compile(
    r'^(' + '|'.join(re.escape(m) for m in sorted(AUGMENTED_MUSCLES, key=len, reverse=True)) + r')_',
    re.I,
)

# ── Canonical metric patterns (specific before general) ────────────────────────
_METRIC_RE = [
    ('inter_slice_dice_pred', re.compile(r'inter_slice_dice_pred',        re.I)),
    ('inter_slice_dice_gt',   re.compile(r'inter_slice_dice_gt',          re.I)),
    ('dice',                  re.compile(r'^(?:lower_)?dice$',            re.I)),
    ('hausdorff',             re.compile(r'hausdorff',                    re.I)),
    ('jaccard',               re.compile(r'jaccard',                      re.I)),
    ('volume_similarity',     re.compile(r'volume_similarity',            re.I)),
    ('false_negative',        re.compile(r'false.?neg|falseNeg',          re.I)),
    ('false_positive',        re.compile(r'false.?pos|falsePo',           re.I)),
    ('bce',                   re.compile(r'\bbce\b|binary_cross_entropy', re.I)),
    ('boundary_iou_3d',       re.compile(r'boundary_iou',                 re.I)),
]


def canonical_metric(col: str):
    # Strip muscle-name prefix then match to a canonical metric name.
    bare = _AUG_PREFIX_RE.sub('', col).rstrip(':')
    for name, pat in _METRIC_RE:
        if pat.search(bare):
            return name
    return None


def extract_muscle(stem: str):
    # Parse muscle name from df_{muscle}_{algo_tag}_augmented filename stem.
    s = stem.lower()
    if s.startswith('df_'):
        s = s[3:]
    if s.endswith('_augmented'):
        s = s[:-10]
    for muscle in sorted(AUGMENTED_MUSCLES, key=len, reverse=True):
        if s.startswith(muscle + '_') or s == muscle:
            return muscle
    return None


print('Helpers defined.')
print('Muscle prefix regex:', _AUG_PREFIX_RE.pattern[:80], '...')

Helpers defined.
Muscle prefix regex: ^(hamstrings|quadriceps|sartorius|gracilis)_ ...


In [30]:
_FLOAT64_MAX = np.finfo(np.float64).max


def process_algorithm(label: str, results_dir: pathlib.Path):
    # Return a summary DataFrame for one algorithm, or None if no CSVs found.
    csv_files = sorted(results_dir.glob('df_*.csv'))
    if not csv_files:
        print(f'  [skip] no CSVs in {results_dir}')
        return None

    rows = []
    for csv_path in csv_files:
        muscle = extract_muscle(csv_path.stem)
        if muscle is None:
            print(f'  [skip] cannot identify muscle in {csv_path.name}')
            continue

        df = pd.read_csv(csv_path)
        metric_vals = {}
        for col in df.columns:
            metric_name = canonical_metric(col)
            if metric_name is None:
                continue
            s = pd.to_numeric(df[col], errors='coerce').to_numpy(dtype=np.float64)
            # Replace inf and extreme sentinel values with NaN
            s = np.where(np.isfinite(s) & (np.abs(s) < _FLOAT64_MAX), s, np.nan)
            if not np.isfinite(s).any():
                print(f'  [all non-finite] {csv_path.name} | {col}')
                continue
            metric_vals[metric_name] = np.nanmean(s, dtype=np.float64)

        # Derived: inter-slice dice ratio (pred / gt)
        pred = metric_vals.get('inter_slice_dice_pred')
        gt   = metric_vals.get('inter_slice_dice_gt')
        if pred is not None and gt is not None and gt > 0:
            metric_vals['inter_slice_dice_ratio'] = pred / gt

        row = {'muscle': muscle}
        row.update(metric_vals)
        rows.append(row)

    if not rows:
        return None

    summary = pd.DataFrame(rows).set_index('muscle')
    summary.insert(0, 'algorithm', label)

    numeric  = summary.select_dtypes(include='number').astype(np.float64)
    overall  = numeric.mean().rename('Overall_Mean')
    overall['algorithm'] = label
    summary  = pd.concat([summary, overall.to_frame().T])
    summary.index.name = 'muscle'
    return summary


print('process_algorithm defined.')

process_algorithm defined.


In [32]:
# ── Algorithm registry ─────────────────────────────────────────────────────────
# Each entry: (display_label, relative_path_to_results_augmented_dir)
REGISTRY = [
    ('Dafne',                        'dafne/codes/results_augmented'),
    ('MuscleMap Thigh',              'muscle_map_thigh/codes/results_augmented'),
    ('MuscleMap WB',                 'muscle_map_wb/codes/results_augmented'),
    ('Hirriririir',                  'multimodal-multiethnic/codes/results_augmented'),
    ('MuSeg',                        'museg/codes/results_augmented'),
    ('MedCLIP-SAMv2',                'medclipsamv2/codes/results_augmented'),
    ('MedCLIP-SAMv2 Text+Boxes',     'medclipsamv2textboxes/augmented_results'),
    ('MedSegDiff',                   'medsegdiff/codes/results_augmented'),
]

print(f'{len(REGISTRY)} algorithms in registry:\n')
for label, rdir in REGISTRY:
    path = EVAL_DIR / rdir
    if path.exists():
        n = len(list(path.glob('df_*.csv')))
        status = f'{n} CSVs'
    else:
        status = 'DIR MISSING'
    print(f'  {label:<35}  {status}')

8 algorithms in registry:

  Dafne                                4 CSVs
  MuscleMap Thigh                      4 CSVs
  MuscleMap WB                         4 CSVs
  Hirriririir                          4 CSVs
  MuSeg                                4 CSVs
  MedCLIP-SAMv2                        DIR MISSING
  MedCLIP-SAMv2 Text+Boxes             2 CSVs
  MedSegDiff                           2 CSVs


In [33]:
# ── Process all algorithms ─────────────────────────────────────────────────────
summaries = {}  # label -> DataFrame

for label, rdir in REGISTRY:
    results_dir = EVAL_DIR / rdir
    print(f'\n── {label} ──')
    df = process_algorithm(label, results_dir)
    if df is None:
        continue
    summaries[label] = df

    num_cols = df.select_dtypes(include='number').columns.tolist()
    display(
        df.reset_index()
        .style
        .format('{:.4f}', subset=num_cols)
        .hide(axis='index')
    )

    safe_name = re.sub(r'[^\w]+', '_', label).strip('_').lower()
    out_path  = SUMMARY_DIR / f'{safe_name}_augmented_avg_metrics.csv'
    df.to_csv(out_path, float_format='%.4f')
    print(f'  Saved -> {out_path}')

print(f'\nProcessed {len(summaries)}/{len(REGISTRY)} algorithms.')


── Dafne ──


muscle,algorithm,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,bce,boundary_iou_3d,inter_slice_dice_pred,inter_slice_dice_gt,inter_slice_dice_ratio
gracilis,Dafne,0.012756,150.553149,0.006507,0.322216,0.931721,0.992819,0.154099,0.021176,0.664141,0.764778,0.868410
hamstrings,Dafne,0.054410,137.967696,0.030198,0.643910,0.933484,0.940975,0.364355,0.030634,0.619700,0.852286,0.727103
quadriceps,Dafne,0.009292,116.311699,0.004776,0.507071,0.993816,0.976819,0.611729,0.017372,0.633905,0.875360,0.724165
sartorius,Dafne,0.001932,152.065949,0.000970,0.655875,0.997052,0.998166,0.076601,0.008558,0.513016,0.831430,0.617029
Overall_Mean,Dafne,0.019597,139.224623,0.010613,0.532268,0.964018,0.977195,0.301696,0.019435,0.607690,0.830963,0.734177


  Saved -> C:\Projects\dissector\eval_notebooks\summary_results_augmented\dafne_augmented_avg_metrics.csv

── MuscleMap Thigh ──


muscle,algorithm,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,bce,boundary_iou_3d,inter_slice_dice_pred,inter_slice_dice_gt,inter_slice_dice_ratio
gracilis,MuscleMap Thigh,0.669621,17.764845,0.530361,0.853405,0.299472,0.324055,0.014635,0.497127,0.804013,0.764778,1.051302
hamstrings,MuscleMap Thigh,0.730031,27.811977,0.599321,0.874050,0.251532,0.245222,0.081938,0.382089,0.851329,0.852286,0.998878
quadriceps,MuscleMap Thigh,0.727781,36.316535,0.601357,0.820045,0.226549,0.251928,0.205877,0.282476,0.872484,0.875360,0.996715
sartorius,MuscleMap Thigh,0.643170,34.949119,0.500622,0.772771,0.365892,0.244014,0.021251,0.448711,0.806797,0.831430,0.970373
Overall_Mean,MuscleMap Thigh,0.692651,29.210619,0.557915,0.830068,0.285861,0.266305,0.080925,0.402601,0.833656,0.830963,1.004317


  Saved -> C:\Projects\dissector\eval_notebooks\summary_results_augmented\musclemap_thigh_augmented_avg_metrics.csv

── MuscleMap WB ──


...

  Saved -> C:\Projects\dissector\eval_notebooks\summary_results_augmented\museg_augmented_avg_metrics.csv

── MedCLIP-SAMv2 ──
  [skip] no CSVs in C:\Projects\dissector\eval_notebooks\medclipsamv2\codes\results_augmented

── MedCLIP-SAMv2 Text+Boxes ──


muscle,algorithm,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,bce,boundary_iou_3d,inter_slice_dice_pred,inter_slice_dice_gt,inter_slice_dice_ratio
gracilis,MedCLIP-SAMv2 Text+Boxes,0.085842,273.982172,0.047395,0.089155,0.033802,0.952316,0.812444,0.037675,0.839643,0.764778,1.097890
sartorius,MedCLIP-SAMv2 Text+Boxes,0.183151,263.199549,0.106066,0.190176,0.041198,0.893393,0.470658,0.073280,0.854404,0.831430,1.027632
Overall_Mean,MedCLIP-SAMv2 Text+Boxes,0.134496,268.590861,0.076731,0.139666,0.037500,0.922854,0.641551,0.055478,0.847023,0.798104,1.062761


  Saved -> C:\Projects\dissector\eval_notebooks\summary_results_augmented\medclip_samv2_text_boxes_augmented_avg_metrics.csv

── MedSegDiff ──


muscle,algorithm,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,bce,boundary_iou_3d,inter_slice_dice_pred,inter_slice_dice_gt,inter_slice_dice_ratio
gracilis,MedSegDiff,0.010139,389.887898,0.005107,0.018084,0.423425,0.994861,2.764725,0.003068,0.132880,0.764778,0.173750
sartorius,MedSegDiff,0.006439,386.475740,0.003234,0.012771,0.482672,0.996751,5.784466,0.003153,0.197969,0.831430,0.238107
Overall_Mean,MedSegDiff,0.008289,388.181819,0.004170,0.015428,0.453049,0.995806,4.274596,0.003111,0.165425,0.798104,0.205928


  Saved -> C:\Projects\dissector\eval_notebooks\summary_results_augmented\medsegdiff_augmented_avg_metrics.csv

Processed 7/8 algorithms.


In [34]:
# ── Combined Overall Means ─────────────────────────────────────────────────────
overall_rows = [
    df.loc[['Overall_Mean']]
    for df in summaries.values()
    if 'Overall_Mean' in df.index
]

if not overall_rows:
    print('No results yet — run the gpu_lambda_column_compare_*_augmented notebooks first.')
else:
    combined = pd.concat(overall_rows)
    combined.index = [row['algorithm'] for _, row in combined.iterrows()]
    combined.index.name = 'algorithm'
    combined = combined.drop(columns='algorithm')
    combined = combined.drop(
        columns=['inter_slice_dice_pred', 'inter_slice_dice_gt'], errors='ignore'
    )

    # Prefer dice as the primary sort key
    if 'dice' in combined.columns:
        combined = combined.sort_values('dice', ascending=False)

    num_cols = combined.select_dtypes(include='number').columns.tolist()
    grad_cols = {col: 'RdYlGn' for col in ['dice', 'jaccard', 'boundary_iou_3d',
                                             'inter_slice_dice_ratio']
                 if col in num_cols}
    grad_cols_r = {col: 'RdYlGn_r' for col in ['hausdorff', 'false_negative',
                                                  'false_positive', 'bce']
                   if col in num_cols}

    def _bold_best(s: pd.Series, higher_is_better: bool) -> list[str]:
        # Bold the best (max or min) value in a column; ties all get bolded.
        best = s.max() if higher_is_better else s.min()
        return ['font-weight: bold' if v == best else '' for v in s]

    styler = (
        combined.reset_index()
        .style
        .format('{:.4f}', subset=num_cols)
        .hide(axis='index')
    )
    for col, cmap in {**grad_cols, **grad_cols_r}.items():
        styler = styler.background_gradient(subset=[col], cmap=cmap, axis=0)
    for col in grad_cols:      # higher is better
        styler = styler.apply(_bold_best, subset=[col], higher_is_better=True)
    for col in grad_cols_r:    # lower is better
        styler = styler.apply(_bold_best, subset=[col], higher_is_better=False)

    display(styler)

    out_combined = SUMMARY_DIR / 'overall_means_augmented.csv'
    combined.to_csv(out_combined, float_format='%.4f')
    print(f'Saved -> {out_combined}')

algorithm,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,bce,boundary_iou_3d,inter_slice_dice_ratio
MuscleMap WB,0.796422,20.594641,0.673355,0.902115,0.133716,0.253258,0.060111,0.484098,1.033390
MuscleMap Thigh,0.692651,29.210619,0.557915,0.830068,0.285861,0.266305,0.080925,0.402601,1.004317
Hirriririir,0.558250,88.145528,0.425790,0.729345,0.322787,0.468779,0.130769,0.248947,1.066139
MuSeg,0.314886,115.402403,0.235169,0.456238,0.691272,0.317666,0.131356,0.130009,0.700634
MedCLIP-SAMv2 Text+Boxes,0.134496,268.590861,0.076731,0.139666,0.037500,0.922854,0.641551,0.055478,1.062761
Dafne,0.019597,139.224623,0.010613,0.532268,0.964018,0.977195,0.301696,0.019435,0.734177
MedSegDiff,0.008289,388.181819,0.004170,0.015428,0.453049,0.995806,4.274596,0.003111,0.205928


Saved -> C:\Projects\dissector\eval_notebooks\summary_results_augmented\overall_means_augmented.csv
